# Week 3 Lab: LoRA — When Less Training Does More

**ECBS5200 — Practical Deep Learning Engineering**

In Weeks 1-2 you fine-tuned all 149M parameters of ModernBERT. That worked,
but it's expensive — every parameter gets updated, every parameter gets saved.
What if you only trained 2% of the parameters and got the same quality?

That's LoRA. Today you'll apply it to the encoder you know, then see what
happens when you apply the same trick to a completely different architecture.

**Time budget:** ~80 minutes.

**How to use this notebook:**
- **GUIDED** cells run as-is. Read them.
- **INTERACTIVE** cells require you to fill in values or write answers.
- `___` is a placeholder that will cause a NameError if not filled in.

## Kaggle setup (do this first!)

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → GPU T4

In [ ]:
import subprocess, sys, os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# HuggingFace config — set BEFORE importing HF libraries
if os.path.exists('/kaggle/working'):
    os.environ['HF_HOME'] = '/kaggle/working/.hf_cache'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.53", "datasets", "accelerate", "scikit-learn",
    "matplotlib", "pandas", "tqdm", "peft",
])
print("Packages installed.")

from huggingface_hub import login
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace.")
else:
    print("WARNING: No HF_TOKEN found. May hit rate limits.")

In [ ]:
import gc
import time
import json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# Load course utilities
REPO_PATH = Path("../..").resolve()
if not (REPO_PATH / "utils" / "data_utils.py").exists():
    REPO_PATH = Path("/tmp/course")
    if not REPO_PATH.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/earino/applied-deep-learning.git", str(REPO_PATH)])
sys.path.insert(0, str(REPO_PATH))

from utils.data_utils import (
    load_course_data, LABEL_LIST, NUM_LABELS, LABEL_TO_ID, ID_TO_LABEL, MAX_LENGTH,
)

In [ ]:
# GUIDED: Device and precision setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    USE_BF16 = cc[0] >= 8
    USE_FP16 = not USE_BF16
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Precision: {'bf16' if USE_BF16 else 'fp16'}")
else:
    USE_BF16, USE_FP16 = False, False
    print("No GPU detected.")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# GUIDED: Load the canonical dataset
print("Loading dataset...")
train_ds, val_ds, _ = load_course_data()
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Classes: {NUM_LABELS}")

---
## Part 1: What Is LoRA? (~3 min)

You covered LoRA in pre-work module 6. Quick recap:

**Full fine-tuning** updates every parameter in the model. For ModernBERT-base,
that's 149M parameters. You need to save all 149M in a checkpoint.

**LoRA** freezes the entire model and injects small trainable matrices
into specific layers. Instead of updating a 768×768 weight matrix directly,
LoRA learns two small matrices (768×16 and 16×768) whose product approximates
the update. Rank 16 means each adapted layer adds only 768×16×2 = 24,576
parameters instead of 589,824.

The question: **does this sacrifice quality?**
---

---
## Part 2: Encoder LoRA (~40 min)

You've been training ModernBERT-base since Week 1. Now apply LoRA to it
instead of updating all 149M parameters.
---

In [ ]:
# GUIDED: Load the base model
ENC_MODEL = "answerdotai/ModernBERT-base"

print(f"Loading {ENC_MODEL}...")
encoder = AutoModelForSequenceClassification.from_pretrained(
    ENC_MODEL,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    attn_implementation="sdpa",
)

total_params = sum(p.numel() for p in encoder.parameters())
print(f"Total parameters: {total_params:,}")

### Explore the model's layers

Before configuring LoRA, you need to know what's inside the model.
LoRA is applied to specific named modules — usually the attention and
feed-forward layers. Run the cell below to see all the layer names.

In [ ]:
# GUIDED: Print all named modules
for name, module in encoder.named_modules():
    if hasattr(module, "weight") and module.weight.dim() >= 2:
        shape = tuple(module.weight.shape)
        print(f"  {name:<55} {str(shape):>15}")

**Study the output above.** You'll see patterns like:
- `model.layers.N.attn.Wqkv` — the attention query/key/value projection
- `model.layers.N.attn.Wo` — the attention output projection
- `model.layers.N.mlp.Wi` — the feed-forward input
- `model.layers.N.mlp.Wo2` — the feed-forward output

These repeat 22 times (one per transformer layer). LoRA will inject
trainable matrices into whichever modules you target.

### INTERACTIVE: Configure LoRA

Fill in the LoRA configuration. You need to make two decisions:

1. **Which modules to target.** Look at the layer names above. You could
   target just the attention layers, or the attention AND feed-forward layers.
   Which do you think will work better? Why? Make a choice and justify it.

2. **What rank to use.** Higher rank = more trainable parameters = more
   capacity, but also more memory and slower training. The rank controls
   the size of the low-rank matrices. What's a reasonable starting point?

In [ ]:
# INTERACTIVE: Fill in the blanks. Justify your choices in a comment.
lora_config = LoraConfig(
    task_type="SEQ_CLS",
    r=___,                     # What rank? Your choice, your reasoning.
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=___,        # Which modules from the list above? A list of strings.
)

In [ ]:
# GUIDED: Apply LoRA to the model
encoder = get_peft_model(encoder, lora_config)
encoder = encoder.to(device)

trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
frozen = total_params - trainable

print(f"Total:     {total_params:,}")
print(f"Trainable: {trainable:,} ({100*trainable/total_params:.2f}%)")
print(f"Frozen:    {frozen:,} ({100*frozen/total_params:.2f}%)")

**Before you move on:** you're about to train ~3.5M parameters out of 149M.
That's roughly 2%. In Week 1, you trained all 149M. Think about what
you expect:

**PREDICTION:** Will LoRA match full fine-tuning? Do better? Do worse?
Write a number — what macro F1 do you expect from encoder LoRA?
(Your Week 1 full FT reference: 56.6% accuracy, 0.209 macro F1)

YOUR PREDICTION:



In [ ]:
# GUIDED: Tokenize the data for training
enc_tokenizer = AutoTokenizer.from_pretrained(ENC_MODEL)

def tokenize_fn(batch):
    return enc_tokenizer(
        batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False,
    )

print("Tokenizing...")
enc_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["text", "label_name"])
enc_val = val_ds.map(tokenize_fn, batched=True, remove_columns=["text", "label_name"])
enc_train = enc_train.rename_column("label", "labels")
enc_val = enc_val.rename_column("label", "labels")
print(f"  Train: {len(enc_train):,}  Val: {len(enc_val):,}")

In [ ]:
# GUIDED: Set up training with HuggingFace Trainer
collator = DataCollatorWithPadding(tokenizer=enc_tokenizer, padding=True,
                                   pad_to_multiple_of=8)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir="encoder_lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=180,
    lr_scheduler_type="linear",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=USE_FP16,
    bf16=USE_BF16,
    seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
)

trainer = Trainer(
    model=encoder,
    args=training_args,
    train_dataset=enc_train,
    eval_dataset=enc_val,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# GUIDED: Train! This takes ~32 minutes on T4.
# Watch the training loss — you'll see it logged every 100 steps.
print("Training encoder LoRA (3 epochs, ~32 min on T4)...")
t0 = time.time()
trainer.train()
enc_train_time = time.time() - t0
print(f"\nDone in {enc_train_time/60:.1f} min")

If training didn't finish in time, set `USE_PRETRAINED = True` below
to load the pre-trained encoder LoRA checkpoint instead.

In [ ]:
USE_PRETRAINED_ENCODER = False  # Set to True if training didn't complete

if USE_PRETRAINED_ENCODER:
    print("Loading pre-trained encoder LoRA from HuggingFace Hub...")
    del encoder, trainer
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    encoder = AutoModelForSequenceClassification.from_pretrained(
        "earino/ecbs5200-week3-encoder-lora",
        attn_implementation="sdpa",
    ).to(device)
    encoder.eval()

    # Re-create trainer just for evaluation
    trainer = Trainer(
        model=encoder,
        args=TrainingArguments(output_dir="/tmp/eval", per_device_eval_batch_size=64,
                               report_to="none"),
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    print("Pre-trained encoder loaded.")

In [ ]:
# GUIDED: Evaluate on the validation set
print("Evaluating encoder LoRA...")
enc_metrics = trainer.evaluate(enc_val)

# Per-class F1
enc_preds = trainer.predict(enc_val)
enc_pred_ids = np.argmax(enc_preds.predictions, axis=-1)
enc_true_ids = enc_preds.label_ids
enc_per_class_f1 = f1_score(enc_true_ids, enc_pred_ids, average=None, zero_division=0,
                             labels=list(range(NUM_LABELS)))

print(f"\n{'='*50}")
print(f"ENCODER LORA RESULTS")
print(f"{'='*50}")
print(f"  Accuracy:        {enc_metrics['eval_accuracy']*100:.1f}%")
print(f"  Macro F1:        {enc_metrics['eval_macro_f1']:.4f}")
print(f"  Zero-F1 classes: {sum(1 for f in enc_per_class_f1 if f == 0.0)}/113")
print(f"\nWeek 1 reference (full fine-tune, all 149M params):")
print(f"  Accuracy: 56.6%   Macro F1: 0.209   Zero-F1: 46")

**Was your prediction right?** LoRA trains 2% of the parameters and gets
_____________ compared to the full fine-tune.

YOUR OBSERVATION:



In [ ]:
# Clean up encoder to free GPU memory for Part 3
del encoder, trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Encoder model unloaded.")

---
## Part 3: A Different Architecture (~15 min)

Everything so far has used ModernBERT — an **encoder** model (149M params).
Encoders read the whole input at once and produce a representation.
They were designed for understanding text.

There's another family of models: **decoders** (GPT, Llama, Qwen).
These read text left-to-right and were designed for generating text.
They're usually bigger — the smallest Qwen is 494M parameters.

Nobody uses decoders for classification. They're for chatbots and code
generation. They're slow. They're overkill.

...right?
---

In [ ]:
# GUIDED: Load a pre-trained decoder (Qwen 0.5B with LoRA, already trained)
DECODER_CHECKPOINT = "earino/ecbs5200-week3-decoder-lora"

print(f"Loading pre-trained decoder from {DECODER_CHECKPOINT}...")
t0 = time.time()
decoder = AutoModelForSequenceClassification.from_pretrained(
    DECODER_CHECKPOINT,
).to(device)
decoder.eval()

dec_params = sum(p.numel() for p in decoder.parameters())
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Parameters: {dec_params:,}")
print(f"\nThis is Qwen2.5-0.5B — a decoder model with 494M parameters.")
print(f"It was trained with LoRA on the SAME data, SAME task, SAME number of epochs.")

In [ ]:
# GUIDED: Tokenize for the decoder
# Decoders need LEFT padding — the classification head reads the LAST
# token, so padding must go on the left side.
dec_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
dec_tokenizer.padding_side = "left"
if dec_tokenizer.pad_token is None:
    dec_tokenizer.pad_token = dec_tokenizer.eos_token

def tokenize_decoder(batch):
    return dec_tokenizer(
        batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False,
    )

print("Tokenizing for decoder...")
dec_val = val_ds.map(tokenize_decoder, batched=True, remove_columns=["text", "label_name"])
dec_val = dec_val.rename_column("label", "labels")

dec_collator = DataCollatorWithPadding(tokenizer=dec_tokenizer, padding=True,
                                       pad_to_multiple_of=8)

### INTERACTIVE: Make your prediction

You have two models trained on the same data for the same task:

| | Encoder (ModernBERT) | Decoder (Qwen 0.5B) |
|---|---|---|
| Parameters | 149M | 494M |
| Trained with LoRA | 2.3% (~3.5M) | 0.46% (~2.3M) |
| Pretraining data | ~2T tokens (English web) | ~18T tokens (multilingual) |
| Architecture | Bidirectional | Left-to-right |

**PREDICTION:** Which model has higher macro F1 on our 113-class task?
By how much? Why?

YOUR PREDICTION:



In [ ]:
# GUIDED: Evaluate the decoder
print("Evaluating decoder...")
dec_trainer = Trainer(
    model=decoder,
    args=TrainingArguments(output_dir="/tmp/dec_eval", per_device_eval_batch_size=64,
                           report_to="none"),
    data_collator=dec_collator,
    compute_metrics=compute_metrics,
)

dec_metrics = dec_trainer.evaluate(dec_val)

dec_preds = dec_trainer.predict(dec_val)
dec_pred_ids = np.argmax(dec_preds.predictions, axis=-1)
dec_true_ids = dec_preds.label_ids
dec_per_class_f1 = f1_score(dec_true_ids, dec_pred_ids, average=None, zero_division=0,
                             labels=list(range(NUM_LABELS)))

### The reveal

In [ ]:
# GUIDED: Side-by-side comparison
enc_acc = enc_metrics["eval_accuracy"]
enc_f1 = enc_metrics["eval_macro_f1"]
enc_zero = sum(1 for f in enc_per_class_f1 if f == 0.0)

dec_acc = dec_metrics["eval_accuracy"]
dec_f1 = dec_metrics["eval_macro_f1"]
dec_zero = sum(1 for f in dec_per_class_f1 if f == 0.0)

print("=" * 60)
print("ENCODER vs DECODER")
print("=" * 60)
print()
print(f"{'Metric':<25} {'Encoder LoRA':>15} {'Decoder LoRA':>15} {'Winner':>10}")
print("-" * 65)
print(f"{'Accuracy':<25} {enc_acc*100:>14.1f}% {dec_acc*100:>14.1f}% "
      f"{'Decoder' if dec_acc > enc_acc else 'Encoder':>10}")
print(f"{'Macro F1':<25} {enc_f1:>15.4f} {dec_f1:>15.4f} "
      f"{'Decoder' if dec_f1 > enc_f1 else 'Encoder':>10}")
print(f"{'Zero-F1 classes':<25} {enc_zero:>12}/113 {dec_zero:>12}/113 "
      f"{'Decoder' if dec_zero < enc_zero else 'Encoder':>10}")
print(f"{'Parameters':<25} {'149M':>15} {'494M':>15}")
print(f"{'Params trained':<25} {'2.3%':>15} {'0.46%':>15}")
print()
print(f"{'TF-IDF + LogReg (ref)':<25} {'54.2%':>15} {'—':>15}")
print(f"{'Full FT 3ep (ref)':<25} {'56.6% / 0.209':>15} {'—':>15}")

---
## Part 4: Where Does the Decoder Win? (~15 min)

The aggregate numbers tell you WHO wins. The per-class numbers tell you WHERE.
---

In [ ]:
# GUIDED: Compute per-class F1 comparison
label_counts = Counter(train_ds["label"])
sorted_counts = sorted(label_counts.values())
q25 = sorted_counts[len(sorted_counts) // 4]

rare_ids = sorted([lid for lid, c in label_counts.items() if c <= q25])
common_ids = sorted([lid for lid in label_counts if lid not in set(rare_ids)])

print(f"Rare classes (count <= {q25}): {len(rare_ids)}")
print(f"Common classes:                {len(common_ids)}")

In [ ]:
# GUIDED: Per-class F1 by frequency tier
for tier_name, tier_ids in [("RARE", rare_ids), ("COMMON", common_ids)]:
    enc_tier = [enc_per_class_f1[i] for i in tier_ids]
    dec_tier = [dec_per_class_f1[i] for i in tier_ids]
    enc_z = sum(1 for f in enc_tier if f == 0.0)
    dec_z = sum(1 for f in dec_tier if f == 0.0)

    print(f"\n{tier_name} classes (n={len(tier_ids)}):")
    print(f"  Encoder LoRA:  mean F1 = {np.mean(enc_tier):.4f}   zero-F1 = {enc_z}")
    print(f"  Decoder LoRA:  mean F1 = {np.mean(dec_tier):.4f}   zero-F1 = {dec_z}")

**Look at the rare class numbers carefully.**

What pattern do you see? Why does one model do better on rare classes
than the other? Write your explanation before clicking the reveal.

YOUR EXPLANATION:



<details>
<summary><b>Click to check your reasoning</b></summary>

For rare classes with only 4-27 training examples, the encoder has no
way to learn useful representations — it's never seen these concepts
before training. The decoder has. During pretraining on 18 trillion
tokens, it encountered these topics in the wild. That prior knowledge
gives the decoder a starting point that no amount of fine-tuning on
4 examples can create from scratch.

The encoder is **blind** to rare classes. The decoder is not.

</details>

In [ ]:
# GUIDED: Scatter plot — encoder vs decoder per-class F1
colors = ["#e74c3c" if i in set(rare_ids) else "#3498db" for i in range(NUM_LABELS)]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(enc_per_class_f1, dec_per_class_f1, c=colors, alpha=0.6, s=25)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, linewidth=1)
ax.set_xlabel("Encoder LoRA (per-class F1)", fontsize=12)
ax.set_ylabel("Decoder LoRA (per-class F1)", fontsize=12)
ax.set_title("Per-Class F1: Encoder vs Decoder", fontsize=13, fontweight="bold")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(handles=[
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c',
               label=f'Rare (n={len(rare_ids)})'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db',
               label=f'Common (n={len(common_ids)})'),
], loc='lower right', fontsize=11)
ax.grid(True, alpha=0.2)

# Add annotation
above_line = sum(1 for i in range(NUM_LABELS)
                 if dec_per_class_f1[i] > enc_per_class_f1[i])
below_line = NUM_LABELS - above_line
ax.text(0.05, 0.92, f"Decoder wins: {above_line} classes\nEncoder wins: {below_line} classes",
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()

**Points above the diagonal** = decoder wins on that class.
**Points below the diagonal** = encoder wins.
**Red points** on the x-axis = rare classes where the encoder scores zero
and the decoder may have some signal.

### INTERACTIVE: Interpret the results

1. What does the decoder know about rare classes that the encoder doesn't?
   Where did that knowledge come from?

2. Look at where the encoder wins (points below the diagonal).
   What kind of classes are those? (Hint: check their training counts.)

3. Given these results, why would anyone deploy the encoder instead of
   the decoder? Think about factors beyond just accuracy and F1.

YOUR ANSWERS:

1.

2.

3.


### INTERACTIVE: What would you try next?

You've seen that the decoder wins on quality and the encoder wins on speed.
If you had one more experiment to run, what would it be? Pick one:

- Change the LoRA rank (higher? lower?) and predict the effect
- Target different modules (attention-only vs attention+FFN)
- Try class weighting on the decoder (it helped the encoder in Week 2...)
- Something else entirely

**Your experiment:** describe what you'd change, what you'd hold constant,
and what you predict would happen.

YOUR ANSWER:



---
## Wrap-up
---

In [ ]:
# GUIDED: Save results for homework
results = {
    "encoder_lora": {
        "accuracy": round(enc_acc, 4),
        "macro_f1": round(enc_f1, 4),
        "zero_f1_classes": enc_zero,
        "per_class_f1": [round(float(f), 4) for f in enc_per_class_f1],
    },
    "decoder_lora": {
        "accuracy": round(dec_acc, 4),
        "macro_f1": round(dec_f1, 4),
        "zero_f1_classes": dec_zero,
        "per_class_f1": [round(float(f), 4) for f in dec_per_class_f1],
    },
    "label_list": LABEL_LIST,
    "label_counts": {str(k): v for k, v in label_counts.items()},
}

with open("week3_lab_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved week3_lab_results.json")
print("Download this file — you'll need it for the homework.")

### What's next (homework)

In the homework you will:
1. **Train the decoder yourself** — configure LoRA on Qwen 0.5B and
   run the full training loop
2. **Try class weighting** on the decoder — does the trick that helped
   the encoder in Week 2 also help the decoder?
3. **Benchmark latency** — measure how fast each model runs
4. **Write your memo** — the central question: given these results,
   what would you deploy and why?